# Project 1



In [ ]:
from pyspark.sql.types import *
import pyspark.sql.functions as f

from pyspark import SparkContext
from pyspark.sql import SparkSession
 
spark = SparkSession.builder.appName("my_project_1").getOrCreate()
sc = spark.sparkContext

In [ ]:
# Read a CSV into a dataframe
# There is a smarter version, that will first check if there is a Parquet file and use it
def load_csv_file(filename, schema):
  # Reads the relevant file from distributed file system using the given schema

  allowed_files = {'Daily program data': ('Daily program data', "|"),
                   'demographic': ('demographic', "|")}

  if filename not in allowed_files.keys():
    print(f'You were trying to access unknown file \"{filename}\". Only valid options are {allowed_files.keys()}')
    return None

  filepath = allowed_files[filename][0]
  dataPath = f"dbfs:/mnt/coursedata2024/fwm-stb-data/{filepath}"
  delimiter = allowed_files[filename][1]

  df = spark.read.format("csv")\
    .option("header","false")\
    .option("delimiter",delimiter)\
    .schema(schema)\
    .load(dataPath)
  return df

# This dict holds the correct schemata for easily loading the CSVs
schemas_dict = {'Daily program data':
                  StructType([
                    StructField('prog_code', StringType()),
                    StructField('title', StringType()),
                    StructField('genre', StringType()),
                    StructField('air_date', StringType()),
                    StructField('air_time', StringType()),
                    StructField('Duration', FloatType())
                  ]),
                'viewing':
                  StructType([
                    StructField('device_id', StringType()),
                    StructField('event_date', StringType()),
                    StructField('event_time', IntegerType()),
                    StructField('mso_code', StringType()),
                    StructField('prog_code', StringType()),
                    StructField('station_num', StringType())
                  ]),
                'viewing_full':
                  StructType([
                    StructField('mso_code', StringType()),
                    StructField('device_id', StringType()),
                    StructField('event_date', IntegerType()),
                    StructField('event_time', IntegerType()),
                    StructField('station_num', StringType()),
                    StructField('prog_code', StringType())
                  ]),
                'demographic':
                  StructType([StructField('household_id',StringType()),
                    StructField('household_size',IntegerType()),
                    StructField('num_adults',IntegerType()),
                    StructField('num_generations',IntegerType()),
                    StructField('adult_range',StringType()),
                    StructField('marital_status',StringType()),
                    StructField('race_code',StringType()),
                    StructField('presence_children',StringType()),
                    StructField('num_children',IntegerType()),
                    StructField('age_children',StringType()), #format like range - 'bitwise'
                    StructField('age_range_children',StringType()),
                    StructField('dwelling_type',StringType()),
                    StructField('home_owner_status',StringType()),
                    StructField('length_residence',IntegerType()),
                    StructField('home_market_value',StringType()),
                    StructField('num_vehicles',IntegerType()),
                    StructField('vehicle_make',StringType()),
                    StructField('vehicle_model',StringType()),
                    StructField('vehicle_year',IntegerType()),
                    StructField('net_worth',IntegerType()),
                    StructField('income',StringType()),
                    StructField('gender_individual',StringType()),
                    StructField('age_individual',IntegerType()),
                    StructField('education_highest',StringType()),
                    StructField('occupation_highest',StringType()),
                    StructField('education_1',StringType()),
                    StructField('occupation_1',StringType()),
                    StructField('age_2',IntegerType()),
                    StructField('education_2',StringType()),
                    StructField('occupation_2',StringType()),
                    StructField('age_3',IntegerType()),
                    StructField('education_3',StringType()),
                    StructField('occupation_3',StringType()),
                    StructField('age_4',IntegerType()),
                    StructField('education_4',StringType()),
                    StructField('occupation_4',StringType()),
                    StructField('age_5',IntegerType()),
                    StructField('education_5',StringType()),
                    StructField('occupation_5',StringType()),
                    StructField('polit_party_regist',StringType()),
                    StructField('polit_party_input',StringType()),
                    StructField('household_clusters',StringType()),
                    StructField('insurance_groups',StringType()),
                    StructField('financial_groups',StringType()),
                    StructField('green_living',StringType())
                  ])
}

# Read demogrphic data


In [ ]:
%%time
# demographic data filename is 'demographic' 
demo_df = load_csv_file('demographic', schemas_dict['demographic'])

# Read Daily program data

In [ ]:
%%time 
from pyspark.sql.functions import col, to_date, date_format, substring, row_number, countDistinct
from pyspark.sql.window import Window

# daily_program data filename is 'Daily program data' 
daily_prog_df = load_csv_file('Daily program data', schemas_dict['Daily program data'])

#we don't want a duplicates rows 
daily_prog_df = daily_prog_df.distinct() 

#we don't want a situations in which there is same program(identified by prog_code) with different title/genre so bellow are the steps : 

# Identify prog_code with multiple distinct titles or genres 
conflicting_prog_codes = daily_prog_df.groupBy('prog_code') \
    .agg(
        countDistinct('title').alias('distinct_title_count'),
        countDistinct('genre').alias('distinct_genre_count') 
    ) \
    .filter((col('distinct_title_count') > 1) | (col('distinct_genre_count') > 1)) \
    .select('prog_code') 

# Filter out conflicting prog_code rows
daily_prog_df_filtered = daily_prog_df.join(conflicting_prog_codes, on='prog_code', how='left_anti') 

# Assign single title and genre for each conflicting prog_code 
windowSpec = Window.partitionBy('prog_code').orderBy('title') #the orderBy doesn't relevant for our needs
single_title_genre_per_progcode = daily_prog_df.join(conflicting_prog_codes, on = 'prog_code') \
                                        .withColumn("row_num", row_number().over(windowSpec)) \
                                        .filter(col("row_num") == 1) \
                                        .select('prog_code', 'title', 'genre') 

# Then, we leave for each confilcting prog_code only the compatible rows(with its decided (title, genre))
compatible_confilcting_progcodes = daily_prog_df.join(conflicting_prog_codes, on='prog_code') \
                                            .join(single_title_genre_per_progcode, on=['prog_code','title','genre']) 
                                                                                                           
# Combine non-conflicting and resolved conflicting households 
daily_prog_df_resolved = daily_prog_df_filtered.union(compatible_confilcting_progcodes) 
#over 

# we use that version of daily_prog_df_resolved only when needed 
daily_prog_df_with_date = daily_prog_df_resolved.withColumn('air_date', to_date(col('air_date'), 'yyyyMMdd'))
daily_prog_df_with_date= daily_prog_df_with_date.filter(daily_prog_df_with_date.air_date.isNotNull())
daily_prog_df_with_date= daily_prog_df_with_date.filter(daily_prog_df_with_date.air_time.isNotNull())
daily_prog_df_with_air_day = daily_prog_df_with_date.withColumn('air_day', date_format(col('air_date'), 'EEEE'))
new_daily_prog_df = daily_prog_df_with_air_day.withColumn('air_hour', substring(col('air_time'), 1, 2))


# Read viewing data

In [ ]:
# Sample of 10 Million viewing entries

dataPath = f"dbfs:/viewing_10M"
viewing10m_df = spark.read.format("csv")\
    .option("header","true")\
    .option("delimiter",",")\
    .schema(schemas_dict['viewing_full'])\
    .load(dataPath) 
# we don't want unnecessary rows 
viewing10m_df = viewing10m_df.distinct() 


# Read reference data

In [ ]:
%%time
# reference data is stored in parquet for your convinence.
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, countDistinct

ref_df = spark.read.parquet('dbfs:/refxml_new_parquet')
print(f'ref_df contains {ref_df.count()} records!')

updated_ref_df = ref_df.withColumn('dma_contains_z', col('dma').contains('z'))  
 
# we don't want a situation in which there is same device to 2 different households or same device with 
# same household (Unnecessary rows) 

new_ref_df = updated_ref_df.dropDuplicates(subset=['device_id']) 

# we don't want a situation in which there is same household with 2 different DMA so bellow are the steps:

# Identify households associated with multiple DMAs
conflicting_households = new_ref_df.groupBy('household_id') \
                        .agg(countDistinct('dma').alias('DMA_count')) \
                        .filter(col('DMA_count') > 1) \
                        .select('household_id') 

# Filter out rows with conflicting household_id and different dma 
ref_df_filtered = new_ref_df.join(conflicting_households, on='household_id', how='left_anti') 

# Assign a single DMA per conflicting household 
windowSpec2 = Window.partitionBy('household_id').orderBy('device_id') #the orderBy doesn't relevant for our needs
single_dma_per_household = new_ref_df.join(conflicting_households, on='household_id') \
                                    .withColumn("row_num", row_number().over(windowSpec2)) \
                                    .filter(col("row_num") == 1) \
                                    .select("household_id", "dma") 
            
# Then, we leave for each confilcting household only the compatible rows(with its decided dma )
compatible_conflicting_households = new_ref_df.join(conflicting_households, on='household_id') \
                                            .join(single_dma_per_household, on=['household_id', 'dma']) 
                                                                                                    
# Combine non-conflicting and resolved conflicting households
ref_df_resolved = ref_df_filtered.union(compatible_conflicting_households) 

#over 



In [ ]:
#cond 1 

from pyspark.sql.functions import count, avg, col 
ds_filtered = viewing10m_df.filter((col('device_id').isNotNull()) & (col('event_date').isNotNull()))  
views_device_day = viewing10m_df.groupBy('device_id', 'event_date').agg(count('*').alias('count'))  
device_avg = views_device_day.groupBy('device_id').agg(avg('count').alias('avg')) 
device_avg_relevant = device_avg.filter(col('avg') > 5) 
relevant_devices = device_avg_relevant.select('device_id').distinct() 
con1_programs = viewing10m_df.join(relevant_devices, on='device_id').select('prog_code').distinct() 
filtered_con1_programs = con1_programs.filter(col('prog_code').isNotNull()) 
con1_records = daily_prog_df_resolved.join(filtered_con1_programs, on='prog_code') 
print(con1_records.count()) 

In [ ]:
#Cond 2

filtered_ref_df = ref_df_resolved.filter(col('dma_contains_z') == 'true') 
filtered_ref_df = filtered_ref_df.filter(col('device_id').isNotNull()) 
relevant_devices_2 = filtered_ref_df.select('device_id').distinct() 
relevant_devices_viewing_2 = viewing10m_df.join(relevant_devices_2, on='device_id') 
relevant_devices_viewing_2 = relevant_devices_viewing_2.filter(col('prog_code').isNotNull()) 
con2_programs = relevant_devices_viewing_2.select('prog_code').distinct() 
con2_records = daily_prog_df_resolved.join(con2_programs, on='prog_code')  
print(con2_records.count()) 


In [ ]:
#cond 3 

relevant_families_3 = demo_df.filter((col('num_adults') < 3) & (col('net_worth') > 8)) 
relevant_families_3 = relevant_families_3.filter(col('household_id').isNotNull()) 
ref_df_3 = ref_df_resolved.filter(col('household_id').isNotNull())
relevant_devices_3 = ref_df_3.join(relevant_families_3, on='household_id').select('device_id').distinct() 
relevant_device_3 = relevant_devices_3.filter(col('device_id').isNotNull()) 
con3_programs = viewing10m_df.join(relevant_devices_3, on='device_id').select('prog_code').distinct() 
con3_programs = con3_programs.filter(col('prog_code').isNotNull()) 
con3_records = daily_prog_df_resolved.join(con3_programs, on='prog_code')
print(con3_records.count()) 


In [ ]:
# Cond 4

relevant4 = new_daily_prog_df.filter(
    ((col('air_day') == 'Saturday') & (col('air_hour').cast('int') <= 19)) |
    ((col('air_day') == 'Friday') & (col('air_hour').cast('int') >= 18))
) 

relevant_prog_code4 = relevant4.select('prog_code').distinct()
relevant_prog_code4 = relevant_prog_code4.filter(col('prog_code').isNotNull())
con4_records = relevant_prog_code4.join(daily_prog_df_resolved, on=('prog_code')) 
# Display the filtered DataFrame 
record_count4 = con4_records.count() 
print(f'The number of records in prog_code_df is: {record_count4}') 

In [ ]:
#cond5 

relevant_families_5 = demo_df.filter(col('household_size') >= 8).select('household_id').distinct() 
relevant_families_5 = relevant_families_5.filter(col('household_id').isNotNull()) 
relevant_devices_5 = ref_df_resolved.join(relevant_families_5, on= 'household_id').select('device_id').distinct() 
relevant_devices_5 = relevant_devices_5.filter(col('device_id').isNotNull()) 
con5_programs = viewing10m_df.join(relevant_devices_5, on= 'device_id').select('prog_code').distinct() 
con5_programs = con5_programs.filter(col('prog_code').isNotNull()) 
con5_records = daily_prog_df_resolved.join(con5_programs, on='prog_code') 
print(con5_records.count()) 


In [ ]:
from pyspark.sql.functions import col, when, avg 

#cond 6 
# Select distinct 'household_id' and 'income' columns 
inc_house = demo_df.select('household_id', 'income').distinct()
inc_house = inc_house.filter(col('household_id').isNotNull() & col('income').isNotNull())

inc_house = inc_house.withColumn(  
    'income', 
    when(col('income') == 'A', 10)
    .when(col('income') == 'B', 11)
    .when(col('income') == 'C', 12)
    .when(col('income') == 'D', 13)
    .otherwise(col('income').cast('int')) 
)
avg_income = inc_house.select(avg('income')).collect()[0][0] 
dev_house = ref_df_resolved.select('device_id', 'household_id').distinct() 
dev_house = dev_house.filter(col('device_id').isNotNull() & col('household_id').isNotNull())
house_dev_num = dev_house.groupBy('household_id').agg(count('*').alias('device_num')) 
relevant6_house = house_dev_num.filter(col('device_num') > 3) 
relevant6_house = relevant6_house.select('household_id') 
relevant6_house_income = inc_house.join(relevant6_house, on='household_id')
relevant6_house_income_filtered = relevant6_house_income.filter(col('income') < avg_income) 
relevant6_house_filtered = relevant6_house_income_filtered.select('household_id') 
relevant_devices6 = relevant6_house_filtered.join(dev_house, on=('household_id'))
relevant_devices6 = relevant_devices6.select('device_id').distinct() 
relevant_devices6 = relevant_devices6.filter(col('device_id').isNotNull()) 
relevant_prog6 = viewing10m_df.join(relevant_devices6, on=('device_id')) 
relevant_prog6 = relevant_prog6.select('prog_code').distinct()
relevant_prog6 = relevant_prog6.filter(col('prog_code').isNotNull())
con6_records = relevant_prog6.join(daily_prog_df_resolved, on=('prog_code'))
con6_programs_count = con6_records.count()
print(f'final: {con6_programs_count}') 

In [ ]:
#cond 7 
from pyspark.sql.functions import array_contains, split, when 

# List of genres to check for
target_genres = ['Hydroplane racing', 'Biathlon', 'Snowmobile', 'Community', 'Agriculture', 'Music'] 

# Split the prog_genres column into an array of genres 
daily_prog_df_7 = daily_prog_df_resolved.withColumn('prog_genres_array', split(col('genre'), ',')) 

# Filter rows where prog_genres_array contains any of the target genres
filtered_prog_df = daily_prog_df_7.filter( 
    sum(when(array_contains(col('prog_genres_array'), genre), 1).otherwise(0) for genre in target_genres) > 0
)
con7_programs = filtered_prog_df.select('prog_code').distinct().filter(col('prog_code').isNotNull())
con7_records = con7_programs.join(daily_prog_df_resolved, on=('prog_code')) 
print(con7_records.count()) 

In [ ]:
#malicious records  
windowSpec = Window.partitionBy('prog_code','title', 'genre', 'air_date', 'air_time', 'duration') 
conditions_union = con1_records.union(con2_records).union(con3_records).union(con4_records).union(con5_records).union(con6_records).union(con7_records) 
#Add a count column to count the occurrences of each record within the partition  
conditions_with_count = conditions_union.withColumn('record_count', count('*').over(windowSpec))  
# Filter out records that appear less than 4 times 
malicious_records = conditions_with_count.filter(col('record_count') >= 4).drop('record_count').distinct() 
# Select the 'prog_code' column and order by ascending lexicographic order
ordered_malicious_progs_df = malicious_records.select('prog_code').distinct().orderBy('prog_code') 
# Display the top 50 malicious program codes with truncate=False 
ordered_malicious_progs_df.show(50, truncate=False) 
ID1 = '000000000' 
ID2 = '000000000'
output_file_name = f"project1_part1_malicious_{'ID1'}_{'ID2'}.parquet"
# Save the malicious records to a parquet file with the specified name
malicious_records.write.mode("overwrite").parquet(output_file_name)  



